<a href="https://colab.research.google.com/github/yuhui-0611/ESAA/blob/main/ESAA_OB_project_for_paper.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **A Predictive Analysis of Real Estate Investment Decisions Using XGBoost**

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Preprocessing

< 전처리 순서 >

1. 각 연도 df 읽기
2. 각 연도 df에서 논문 기초통계량 17개 변수만 추출
3. 컬럼명을 분석용 공통 이름으로 변경
4. 연도 변수 추가
5. 2019~2025 데이터 병합
6. 코드형 변수 값 분포 확인
7. 논문 기준 더미 변수 생성
8. 불필요한 raw 변수 제거
9. 전체 데이터 결측치 확인
10. 전체 데이터 자료형 확인 및 숫자형 변환
11. 최종 분석용 데이터 완성

## **1. 각 연도 df 읽기**
   - 2019~2025년 csv 읽기
   - data_dict에 저장

In [ ]:
base_path = '/content/drive/MyDrive/ESAA_OB_pj/논문/가구데이터'

# MDIS 자료는 보통 utf-8이 아니라 cp949 또는 euc-kr 인코딩인 경우가 많음
def read_csv_korean(file_path):
    try:
        df = pd.read_csv(file_path, encoding='cp949')
    except UnicodeDecodeError:
        df = pd.read_csv(file_path, encoding='euc-kr')

    return df

In [ ]:
data_dict = {}

for year in range(2019, 2026):
    file_path = f'{base_path}/{year}.csv'
    df = read_csv_korean(file_path)
    data_dict[year] = df
    print(f'{year}년 데이터 읽기 완료 | shape: {df.shape}')

2019년 데이터 읽기 완료 | shape: (18406, 159)
2020년 데이터 읽기 완료 | shape: (18064, 159)
2021년 데이터 읽기 완료 | shape: (18187, 160)
2022년 데이터 읽기 완료 | shape: (17954, 160)
2023년 데이터 읽기 완료 | shape: (18094, 160)
2024년 데이터 읽기 완료 | shape: (18314, 162)
2025년 데이터 읽기 완료 | shape: (18664, 160)


In [ ]:
# 2019~2025년 데이터 컬럼명 확인

for year in range(2019, 2026):
    file_path = f'{base_path}/{year}.csv'

    try:
        df = pd.read_csv(file_path, encoding='cp949')
    except UnicodeDecodeError:
        df = pd.read_csv(file_path, encoding='euc-kr')

    print('=' * 80)
    print(f'{year}년 데이터')
    print(f'데이터 크기: {df.shape}')
    print(f'컬럼 개수: {len(df.columns)}')
    print()
    print(df.columns.tolist())
    print()

2019년 데이터
데이터 크기: (18406, 159)
컬럼 개수: 159

['조사연도', 'MD제공용_가구고유번호', '가중값', '수도권여부', '가구주_성별코드', '가구원수', '노인가구코드', '조손가구코드', '한부모가구코드', '다문화가구코드', '보완_등록장애인유무_가구구분코드', '가구주_교육정도_학력코드', '가구주_교육정도_수학구분코드', '가구주_교육정도_통합코드', '가구주_동거여부', '가구주_산업대분류코드', '가구주_직업대분류코드', '가구주_만연령', '가구주연령_10세단위코드', '가구주_종사상지위코드', '보도용_가구주_종사상지위코드', '가구주_혼인상태코드', '입주형태코드', '입주형태통합코드', '전용면적규모코드', '주택종류통합코드', '부채보유여부', '보완_소득5분위코드', '보완_소득10분위코드', '보도용_보완_소득계층구간코드', '자산총액5분위코드', '자산총액10분위코드', '순자산5분위코드', '순자산10분위코드', '자산', '자산_금융자산', '자산_금융자산_저축금액', '자산_금융자산_저축_적립예치식저축금액', '자산_금융자산_저축_적립예치식저축_수시적립예치식저축금액', '자산_금융자산_저축_적립예치식저축_저축성보장성보험금액', '자산_금융자산_저축_적립예치식저축_주식채권펀드금액', '자산_금융자산_저축_기타저축금액', '자산_금융자산_현거주지전월세보증금', '자산_실물자산', '자산_실물자산_부동산금액', '자산_실물자산_부동산_거주주택금액', '자산_실물자산_부동산_거주주택이외부동산금액', '자산_실물자산_부동산_계약금중도금납입금액', '자산_실물자산_기타실물자산', '자산_실물자산_기타실물자산_자동차금액', '자산_실물자산_기타실물자산_기타금액', '자산_실물자산_기타실물자산_기타_자동차이외기타실물자산', '자산_실물자산_기타실물자산_기타_권리금', '부채', '부채_금융부채', '부채_금융부채_담보대출금액', '부채_금융부채_담보대출_대출용도_거주주택구입금액', '부채_금융부채_담보대출_대출용

## **2. 각 연도 df에서 논문 기초통계량 17개 변수만 추출**
   - col_map 기준으로 후보 컬럼 중 존재하는 컬럼 선택
   - 코드북이랑 대응되는 컬럼 추출

In [ ]:
col_map = {
    'investment_decision': ['여유자금부동산투자여부'],
    'sma': ['수도권여부'],
    'head_gender': ['가구주_성별코드'],
    'college_or_higher_raw': ['가구주_교육정도_통합코드'],
    'own_house_raw': ['입주형태코드', '입주형태통합코드'],
    'income_decile': ['보완_소득10분위코드', '소득10분위코드(보완)(2017년~)'],
    'household_members': ['가구원수'],
    'head_age': ['가구주_만연령'],
    'financial_assets': ['자산_금융자산'],
    'real_assets': ['자산_실물자산'],
    'secured_loan': ['부채_금융부채_담보대출금액'],
    'unsecured_loan': ['부채_금융부채_신용대출금액'],
    'principal_interest_repayment': ['원리금상환금액'],
    'net_worth': ['순자산'],
    'current_income': ['보완_경상소득', '경상소득(보완)'],
    'expenditure': ['지출_소비지출비'],
    'repayment_to_income_ratio': ['원리금상환_생계부담_소득대비상환액비중']
}

In [ ]:
# col_map에 들어간 후보 컬럼들이 각 연도에 존재하는지 확인

exist_check = []

for year, df in data_dict.items():
    for analysis_var, candidate_cols in col_map.items():

        matched_col = None

        for col in candidate_cols:
            if col in df.columns:
                matched_col = col
                break

        exist_check.append({
            'year': year,
            'analysis_var': analysis_var,
            'matched_column': matched_col,
            'exists': matched_col is not None
        })

exist_df = pd.DataFrame(exist_check)

exist_df

,year,analysis_var,matched_column,exists
0,2019,investment_decision,여유자금부동산투자여부,True
1,2019,sma,수도권여부,True
2,2019,head_gender,가구주_성별코드,True
3,2019,college_or_higher_raw,가구주_교육정도_통합코드,True
4,2019,own_house_raw,입주형태코드,True
...,...,...,...,...
114,2025,principal_interest_repayment,원리금상환금액,True
115,2025,net_worth,순자산,True
116,2025,current_income,경상소득(보완),True
117,2025,expenditure,지출_소비지출비,True


In [ ]:
# 존재하지 않는 변수 확인

missing_df = exist_df[exist_df['exists'] == False]

missing_df

,year,analysis_var,matched_column,exists


- 존재하지 않는 변수 없음 -> 대응되는 컬럼 추출 완료
> col_map에 넣은 후보 컬럼들이 2019~2025년 모든 데이터에서 하나씩 정상적으로 매칭됐다는 뜻

## **3. 컬럼명을 분석용 공통 이름으로 변경**
   - 예: 여유자금부동산투자여부 → investment_decision
   - 예: 보완_경상소득 / 경상소득(보완) → current_income

## **4. 연도 변수 추가**
   - year = 2019, 2020, ..., 2025

## **5. 2019~2025 데이터 병합**
   - df_all 생성

In [ ]:
# 여러 후보 컬럼 중 실제 존재하는 컬럼을 찾는 함수

def get_existing_col(df, candidates):
    for col in candidates:
        if col in df.columns:
            return col
    return None

In [ ]:
# 각 연도 df에서 17개 변수만 추출하고 공통 컬럼명으로 변경

processed_dfs = []

for year, df in data_dict.items():
    temp = pd.DataFrame()

    for analysis_var, candidate_cols in col_map.items():
        matched_col = get_existing_col(df, candidate_cols)
        temp[analysis_var] = df[matched_col]

    temp['year'] = year

    processed_dfs.append(temp)

df_all = pd.concat(processed_dfs, ignore_index=True)

print(df_all.shape)
df_all.head()

(127683, 18)


,investment_decision,sma,head_gender,college_or_higher_raw,own_house_raw,income_decile,household_members,head_age,financial_assets,real_assets,secured_loan,unsecured_loan,principal_interest_repayment,net_worth,current_income,expenditure,repayment_to_income_ratio,year
0,1,G1,1,G4,2,D04,6,56,12460,32000,4000,7000,1285,25620,3360,4044,NaN,2019
1,2,G1,2,G2,2,D05,2,64,38310,48175,0,0,0,79485,4090,3125,NaN,2019
2,2,G1,1,G3,1,D01,2,84,200,20150,0,0,0,20350,774,1350,NaN,2019
3,2,G1,1,G3,1,D08,4,57,2268,44230,10000,0,1430,36198,7411,3070,NaN,2019
4,1,G1,1,G4,1,D09,5,56,25674,132500,23000,900,1085,134274,10219,4940,NaN,2019


- 3~5 단계 한 번에 시행

## **6. 코드형 변수 값 분포 확인**
   - investment_decision, sma, head_gender, college_or_higher_raw, own_house_raw, income_decile 확인

In [ ]:
# 코드형 변수 값 분포 확인

code_cols = [
    'investment_decision',
    'sma',
    'head_gender',
    'college_or_higher_raw',
    'own_house_raw',
    'income_decile'
]

for col in code_cols:
    print('=' * 80)
    print(col)
    print(df_all[col].value_counts(dropna=False).sort_index())

investment_decision
investment_decision
1    61246
2    66437
Name: count, dtype: int64
sma
sma
G1    41431
G2    86252
Name: count, dtype: int64
head_gender
head_gender
1    92407
2    35276
Name: count, dtype: int64
college_or_higher_raw
college_or_higher_raw
G1    24374
G2    13876
G3    41312
G4    48121
Name: count, dtype: int64
own_house_raw
own_house_raw
1    78890
2    14823
3    23489
4     1802
5     8679
Name: count, dtype: int64
income_decile
income_decile
D01    16620
D02    15516
D03    13898
D04    12974
D05    12428
D06    11914
D07    11403
D08    11069
D09    10902
D10    10959
Name: count, dtype: int64


## **7. 논문 기준 더미 변수 생성**
   - 투자=1
   - 수도권=1
   - 남자=1
   - 대학졸업 이상=1
   - 자가=1
   - 소득10분위는 숫자형 1~10으로 변환

> 코드북이랑 비교한 결과

investment_decision: 1 = 투자, 2 = 비투자

sma: G1 = 수도권, G2 = 비수도권

head_gender: 1 = 남자, 2 = 여자

college_or_higher_raw: G4 = 대학졸업 이상

own_house_raw: 1 = 자가

income_decile: D01 ~ D10 = 소득 1 ~ 10분위

In [ ]:
df_all['investment_decision'] = (df_all['investment_decision'] == 1).astype(int)

df_all['sma'] = (df_all['sma'] == 'G1').astype(int)

df_all['head_gender'] = (df_all['head_gender'] == 1).astype(int)

df_all['college_or_higher'] = (df_all['college_or_higher_raw'] == 'G4').astype(int)

df_all['own_house'] = (df_all['own_house_raw'] == 1).astype(int)

df_all['income_decile'] = df_all['income_decile'].str.replace('D', '', regex=False).astype(int)

In [ ]:
# 더미 변수 변환 결과 확인

check_cols = [
    'investment_decision',
    'sma',
    'head_gender',
    'college_or_higher',
    'own_house',
    'income_decile'
]

for col in check_cols:
    print('=' * 80)
    print(col)
    print(df_all[col].value_counts(dropna=False).sort_index())

investment_decision
investment_decision
0    66437
1    61246
Name: count, dtype: int64
sma
sma
0    86252
1    41431
Name: count, dtype: int64
head_gender
head_gender
0    35276
1    92407
Name: count, dtype: int64
college_or_higher
college_or_higher
0    79562
1    48121
Name: count, dtype: int64
own_house
own_house
0    48793
1    78890
Name: count, dtype: int64
income_decile
income_decile
1     16620
2     15516
3     13898
4     12974
5     12428
6     11914
7     11403
8     11069
9     10902
10    10959
Name: count, dtype: int64


- 변환 완료

## **8. 불필요한 raw 변수 제거**
   - college_or_higher_raw
   - own_house_raw 등

In [ ]:
df_all = df_all.drop(columns=[
    'college_or_higher_raw',
    'own_house_raw'
])

- investment_decision, sma, head_gender, income_decile은 기존 컬럼을 그대로 변환했기 때문에 제거하면 안됨

## **9. 전체 데이터 결측치 확인**
   - 변수별 결측 개수 확인
   - 결측 처리 방식 결정

In [ ]:
missing_summary = df_all.isna().sum().sort_values(ascending=False)
missing_summary

,0
repayment_to_income_ratio,105707
investment_decision,0
sma,0
head_gender,0
household_members,0
income_decile,0
financial_assets,0
real_assets,0
secured_loan,0
head_age,0


## **10. 전체 데이터 자료형 확인 및 숫자형 변환**
   - 금액 변수, 소득 변수, 지출 변수 numeric 처리



In [ ]:
df_all.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 127683 entries, 0 to 127682
Data columns (total 18 columns):
 #   Column                        Non-Null Count   Dtype  
---  ------                        --------------   -----  
 0   investment_decision           127683 non-null  int64  
 1   sma                           127683 non-null  int64  
 2   head_gender                   127683 non-null  int64  
 3   income_decile                 127683 non-null  int64  
 4   household_members             127683 non-null  int64  
 5   head_age                      127683 non-null  int64  
 6   financial_assets              127683 non-null  int64  
 7   real_assets                   127683 non-null  int64  
 8   secured_loan                  127683 non-null  int64  
 9   unsecured_loan                127683 non-null  int64  
 10  principal_interest_repayment  127683 non-null  int64  
 11  net_worth                     127683 non-null  int64  
 12  current_income                127683 non-nul

- repayment_to_income_ratio
  - 정수형 컬럼에 NaN이 섞이면 보통 float64로 처리

# **결측 있는거 어떻게 처리할지 결정 및 채우기 해주시면 될 것 같습니다!!!**

In [ ]:
# 결측치가 있는 행 제외한 데이터 생성

df_no_missing = df_all.dropna().copy()

print('원래 데이터 크기:', df_all.shape)
print('결측 제거 후 데이터 크기:', df_no_missing.shape)

# 결측 제거 후 기초통계량 확인

df_no_missing.describe().T

원래 데이터 크기: (127683, 18)
결측 제거 후 데이터 크기: (21976, 18)


,count,mean,std,min,25%,50%,75%,max
investment_decision,21976.0,0.580815,0.493437,0.0,0.00,1.0,1.00,1.0
sma,21976.0,0.363442,0.481001,0.0,0.00,0.0,1.00,1.0
head_gender,21976.0,0.811931,0.390776,0.0,1.00,1.0,1.00,1.0
income_decile,21976.0,6.507417,2.604125,1.0,4.00,7.0,9.00,10.0
household_members,21976.0,2.674008,1.207835,1.0,2.00,3.0,4.00,7.0
head_age,21976.0,52.073398,12.931919,19.0,42.00,52.0,61.00,93.0
financial_assets,21976.0,14779.329587,44345.624036,0.0,3393.50,8200.0,17294.75,5063100.0
real_assets,21976.0,49659.358664,109709.569183,0.0,6578.25,26443.0,59200.00,10683000.0
secured_loan,21976.0,6615.826857,16647.585544,0.0,0.00,1656.0,8100.00,1000000.0
unsecured_loan,21976.0,1258.051920,4235.701020,0.0,0.00,0.0,600.00,125000.0


## **11. 최종 분석용 데이터 완성**
   - y = investment_decision
   - X = 나머지 설명변수